# Model comparison across runs

Exploration only — loads `metrics.json` and `predictions.csv` from every run in
`experiments/runs/`. The aggregated report is `results/SUMMARY.md`, generated by
`python -m src.summarize`.

Run this notebook from `GroupA/daniela`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import iter_runs

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

runs = list(iter_runs())
print("Runs found:", [r["run_id"] for r in runs])


In [ ]:
metric_keys = ["MAE", "RMSE", "R2", "pearson", "spearman",
               "cv_mae_mean", "cv_mae_std", "n"]
rows = []
for run in runs:
    for target, models in run["metrics"].items():
        for model, values in models.items():
            rows.append({"run": run["run_id"], "target": target, "model": model,
                         **{key: values.get(key) for key in metric_keys}})
metrics = pd.DataFrame(rows)
metrics.head()


In [ ]:
best = (metrics.sort_values(["run", "target", "cv_mae_mean"])
        .groupby(["run", "target"], as_index=False)
        .first())
best[["run", "target", "model", "MAE", "RMSE", "R2", "pearson", "spearman"]].round(3)


In [ ]:
best_mae = best.pivot(index="target", columns="run", values="MAE")
best_r2 = best.pivot(index="target", columns="run", values="R2")
print("Best-model MAE")
display(best_mae.round(3))
print("Best-model R²")
display(best_r2.round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
metrics.pivot_table(index="model", columns="target", values="MAE", aggfunc="mean").plot(
    kind="bar", ax=ax
)
ax.set_ylabel("Mean MAE across runs")
ax.set_title("Model comparison (all runs)")
ax.legend(title="Target", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(metrics.pivot_table(index=["run", "model"], columns="target", values="R2"),
            annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("R² by run / model / target")
fig.tight_layout()


In [ ]:
TARGET = "Note composite_WPPSI"

fig, ax = plt.subplots(figsize=(7, 6))
for run in runs:
    path = run["dir"] / "predictions.csv"
    if not path.exists():
        continue
    pred = pd.read_csv(path)
    true_col, pred_col = f"true_{TARGET}", f"predicted_{TARGET}"
    if true_col in pred and pred_col in pred:
        ax.scatter(pred[true_col], pred[pred_col], alpha=0.55,
                   label=run["run_id"])
ax.set_xlabel("Actual")
ax.set_ylabel("Predicted")
ax.set_title(f"Out-of-fold predictions — {TARGET}")
ax.legend()
fig.tight_layout()


In [ ]:
summary = (metrics.sort_values(["run", "target", "cv_mae_mean"])
           .groupby(["run", "target"], as_index=False)
           .first()[["run", "target", "model", "MAE", "R2", "cv_mae_mean", "cv_mae_std"]])
summary.round(3)
